# 第13章 RAG (RAG搭載モデルを検証)

## 13.3 RAG 向けに LLM を指示チューニングする

### 13.3.2 指示チューニングしたモデルを LangChain で使う

#### 環境の準備

In [1]:
!pip install 'datasets<4.0.0' transformers[torch,sentencepiece] langchain langchain-community langchain-huggingface faiss-cpu jq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 773.8/773.8 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 79.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 554.9/554.9 kB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.3
    Uninstalling langchain-core-1.4.3:
      Successfully uninstalled langchain-core-1.4.3
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling

In [2]:
from transformers.trainer_utils import set_seed
from google.colab import drive

set_seed(42)
drive.mount("drive")

Mounted at drive


#### Chat Modelの作成

In [3]:
import torch
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFacePipeline,
)
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    pipeline,
)

# Hugging Face Hubにおけるモデル名を指定
model_name = "llm-book/Swallow-7b-hf-oasst1-21k-ja-aio-retriever"

# モデルを読み込む
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto",
)

# トークナイザを読み込む
tokenizer = AutoTokenizer.from_pretrained(model_name)

# テキスト生成用のパラメータを指定
generation_config = {
    "max_new_tokens": 32,
    "do_sample": False,
    "temperature": None,
    "top_p": None,
}

# テキスト生成を行うパイプラインを作成
text_generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    device_map="auto",
    **generation_config,
)

# パイプラインからLangChainのLLMコンポーネントを作成
llm = HuggingFacePipeline(pipeline=text_generation_pipeline)

# LLMコンポーネントを元にChat Modelコンポーネントを作成
chat_model = ChatHuggingFace(llm=llm, tokenizer=tokenizer)

config.json:   0%|          | 0.00/770 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/914k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'top_p', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


#### Embedding Modelの作成

In [4]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings

# Hugging Face Hubにおけるモデル名を指定
embedding_model_name = "BAAI/bge-m3"

# モデル名からEmbedding Modelを初期化
embedding_model = HuggingFaceEmbeddings(
    model_name=embedding_model_name,
    model_kwargs={"model_kwargs": {"torch_dtype": torch.float16}},
)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

#### データストアの構築

In [5]:
from datasets import load_dataset
ds = load_dataset(
    "singletongue/wikipedia-utils",
    "passages-c400-jawiki-20240401",
)

README.md:   0%|          | 0.00/6.75k [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/300M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/283M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/260M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/252M [00:00<?, ?B/s]

...

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/240M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/238M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/237M [00:00<?, ?B/s]

passages-c400-jawiki-20240401/train-0000(…):   0%|          | 0.00/235M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5807053 [00:00<?, ? examples/s]

In [6]:
print(ds["train"])
print(ds["train"].column_names)
print(ds["train"][0])

Dataset({
    features: ['id', 'pageid', 'revid', 'title', 'section', 'text'],
    num_rows: 5807053
})
['id', 'pageid', 'revid', 'title', 'section', 'text']
{'id': 1, 'pageid': 5, 'revid': 99347164, 'title': 'アンパサンド', 'section': '__LEAD__', 'text': 'アンパサンド(&, 英語: ampersand)は、並立助詞「...と...」を意味する記号である。ラテン語で「...と...」を表す接続詞 "et" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、"et" の合字であることが容易にわかる字形を使用している。'}


In [7]:
documents_str = ds["train"]["text"]
print(len(documents_str))
print(documents_str[0])

5807053
アンパサンド(&, 英語: ampersand)は、並立助詞「...と...」を意味する記号である。ラテン語で「...と...」を表す接続詞 "et" の合字を起源とする。現代のフォントでも、Trebuchet MS など一部のフォントでは、"et" の合字であることが容易にわかる字形を使用している。


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# 文書を指定した文字数で分割するText Splitterを初期化
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,  # 分割する最大文字数
    chunk_overlap=100,  # 分割された文書間で重複させる最大文字数
    add_start_index=True,  # 元の文書における開始位置の情報を付与
)

documents = [Document(page_content=documents_str)]

# 文書の分割を実行
split_documents = text_splitter.split_documents(documents)

# 分割後の文書数を確認
print(len(split_documents))

ValidationError: 1 validation error for Document
page_content
  Input should be a valid string [type=string_type, input_value=['アンパサンド(&, ...ューアルした。'], input_type=list]
    For further information visit https://errors.pydantic.dev/2.12/v/string_type

#### 検索対象の文書のベクトルインデックスの作成

In [ ]:
from langchain_community.vectorstores import FAISS

# 分割後の文書と文埋め込みモデルを用いて、Faissのベクトルインデックスを作成
vectorstore = FAISS.from_documents(split_documents, embedding_model)

# ベクトルインデックスに登録された文書数を確認
print(vectorstore.index.ntotal)

#### Retrieverコンポーネントの作成

In [ ]:
# ベクトルインデックスを元に文書の検索を行うRetrieverを初期化
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [ ]:
# 動作確認
test_retriever_doc = retriever.invoke("日本の神は？")
print(test_retriever_doc[0].page_content)

#### RAGのChainの構築と実行

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 任意のqueryからメッセージを構築するPrompt Templateを作成
rag_prompt_text = (
    "あなたには今からクイズに答えてもらいます。"
    "問題を与えますので、その解答のみを簡潔に出力してください。\n"
    "また解答の参考になりうるテキストを与えます。"
    "解答を含まない場合もあるのでその場合は無視してください。\n\n"
    "---\n{context}\n---\n\n問題: {query}"
)
rag_prompt_template = ChatPromptTemplate.from_messages(
    [("user", rag_prompt_text)]
)

In [ ]:
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

def format_documents_func(documents: list[Document]) -> str:
    """文書のリストを改行で連結した一つの文字列として返す"""
    return "\n\n".join(
        document.page_content for document in documents
    )

# 定義した関数の処理を行うRunnableを作成
format_documents = RunnableLambda(format_documents_func)

In [ ]:
from langchain_core.prompt_values import ChatPromptValue

def chat_model_resp_only_func(
    chat_prompt_value: ChatPromptValue,
) -> str:
    """chat_modelにchat_prompt_valueを入力し、
    出力からモデルの応答部分のみを文字列で返す"""
    chat_prompt = chat_model._to_chat_prompt(
        chat_prompt_value.messages
    )
    chat_output_message = chat_model.invoke(chat_prompt_value)
    response_text = chat_output_message.content[len(chat_prompt) :]
    return response_text

# 定義した関数の処理を行うRunnableを作成
chat_model_resp_only = RunnableLambda(chat_model_resp_only_func)

In [ ]:
from langchain_core.runnables import RunnablePassthrough

# RAGの一連の処理を行うChainを作成
rag_chain = (
    {
        "context": retriever | format_documents,
        "query": RunnablePassthrough(),
    }
    | rag_prompt_template
    | chat_model_resp_only
)

### (EXP) AI王データセットを使用して、検証

#### データセットの準備


In [ ]:
from datasets import load_dataset
# Hugging Face Hubのllm-book/aio-retrieverのリポジトリから
# AI王データセットを読み込む
dataset = load_dataset(
    "llm-book/aio-retriever",
    trust_remote_code=True
)

In [ ]:
# 読み込まれたデータセットの形式と事例数を確認
print(dataset)

print(dataset)

print(dataset["validation"])

print(dataset["validation"][0])


#### 検証の準備

In [ ]:
from datasets import Dataset
from tqdm.notebook import tqdm

def evaluate_rag(
    rag_chain,
    dataset: Dataset,
    limit: int = 100
) -> tuple[list[str], list[list[str]], float]:
    """RAGチェーンを用いてデータセットの各問題に回答し、正解率を算出"""
    pred_answers = []
    gold_answers = []
    num_correct = 0

    # 全件実行すると時間がかかるため、指定した件数(limit)で評価
    for example in tqdm(dataset.select(range(limit))):
        question = example["question"]
        # 正解のリスト（別解が含まれる場合があるためリスト形式）
        gold_answer_list = example["answers"]

        # RAGチェーンに問題を入力し、出力を得る
        pred_answer = rag_chain.invoke(question)  # templateでquestionのみを引数に持つように調整済み

        # モデルの答えが正解リストのいずれかと完全に一致していれば正答とカウント
        if pred_answer in gold_answer_list:
            num_correct += 1

        # モデルの答えと正解リストをそれぞれ追加
        pred_answers.append(pred_answer)
        gold_answers.append(gold_answer_list)

    # 正解率を計算
    accuracy = num_correct / len(pred_answers)

    return pred_answers, gold_answers, accuracy

In [ ]:
# 構築したRAGチェーンを使って評価（時間短縮のため100件で検証）
pred_answers, gold_answers, accuracy = evaluate_rag(
    rag_chain, dataset["validation"], limit=100
)

print(f"正解率: {accuracy:.1%}")

In [ ]:
# 構築したRAGチェーンを使って評価（時間短縮のため100件で検証）
pred_answers, gold_answers, accuracy = evaluate_rag(
    rag_chain, dataset["train"], limit=100
)

print(f"正解率: {accuracy:.1%}")